#### Deep Learning Models

In [ ]:
import os
import numpy as np
from scipy.io import loadmat

all_windows = []
all_labels = []
all_subjects = []

dataset_dir = "raw_data"

for participant in participants:

    mat_file = os.path.join(
        dataset_dir,
        f"Data_Preprocessed_{participant}",
        f"Data_Preprocessed_{participant}.mat"
    )

    print(f"Processing {participant}")

    data = loadmat(mat_file)

    joined = data["joined_data"]
    labels = data["labels_selfassessment"]

    for video_idx in range(joined.shape[1]):

        signal = joined[0, video_idx]

        # Skip recordings containing NaNs
        if np.isnan(signal).any():
            continue

        label = labels[0, video_idx][0]

        windows = create_windows(
            signal,
            window_size=128,
            step_size=64
        )

        for window in windows:

            # Binary Valence Classification
            target = int(label[0] >= 5)

            all_windows.append(window)
            all_labels.append(target)
            all_subjects.append(participant)

print("Extraction Complete")

Processing P01
Processing P07
Processing P08
Processing P12
Processing P13
Processing P14
Processing P15
Processing P22
Processing P23
Processing P24
Processing P25
Processing P28
Processing P36
Processing P37
Processing P40
Extraction Complete


In [ ]:
X_dl = np.array(all_windows, dtype=np.float32)
y_dl = np.array(all_labels, dtype=np.int32)
groups = np.array(all_subjects)

print("X_dl shape:", X_dl.shape)
print("y_dl shape:", y_dl.shape)
print("groups shape:", groups.shape)

X_dl shape: (153540, 128, 17)
y_dl shape: (153540,)
groups shape: (153540,)


In [ ]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X_dl, y_dl, groups)
)

X_train_dl = X_dl[train_idx]
X_test_dl = X_dl[test_idx]

y_train_dl = y_dl[train_idx]
y_test_dl = y_dl[test_idx]

print(X_train_dl.shape)
print(X_test_dl.shape)

(136406, 128, 17)
(17134, 128, 17)


In [ ]:
mean = X_train_dl.mean(axis=(0,1), keepdims=True)
std = X_train_dl.std(axis=(0,1), keepdims=True)

X_train_dl = (X_train_dl - mean) / (std + 1e-8)
X_test_dl = (X_test_dl - mean) / (std + 1e-8)

In [ ]:
print(X_train_dl.shape)
print(X_test_dl.shape)

(136406, 128, 17)
(17134, 128, 17)


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv1D,
    MaxPooling1D,
    BatchNormalization,
    GlobalAveragePooling1D,
    Dense,
    Dropout
)

cnn_model = Sequential([

    Conv1D(
        filters=64,
        kernel_size=3,
        activation='relu',
        input_shape=(128,17)
    ),

    BatchNormalization(),

    MaxPooling1D(pool_size=2),

    Conv1D(
        filters=128,
        kernel_size=3,
        activation='relu'
    ),

    BatchNormalization(),

    MaxPooling1D(pool_size=2),

    GlobalAveragePooling1D(),

    Dense(64, activation='relu'),

    Dropout(0.3),

    Dense(1, activation='sigmoid')
])

cnn_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

cnn_model.summary()

d:\VIT_Projects\amigos\amvenv\lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 126, 64)        │         3,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 126, 64)        │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 63, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 61, 128)        │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 61, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 30, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 37,121 (145.00 KB)

 Trainable params: 36,737 (143.50 KB)

 Non-trainable params: 384 (1.50 KB)

In [ ]:
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau
)

cnn_callbacks = [

    EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    ),

    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2
    ),

    ModelCheckpoint(
        "Models/cnn_best.keras",
        save_best_only=True,
        monitor="val_accuracy"
    )
]

In [ ]:
cnn_history = cnn_model.fit(
    X_train_dl,
    y_train_dl,
    validation_data=(X_test_dl, y_test_dl),
    epochs=20,
    batch_size=256,
    callbacks=cnn_callbacks
)

Epoch 1/20
533/533 ━━━━━━━━━━━━━━━━━━━━ 14s 23ms/step - accuracy: 0.8091 - loss: 0.4233 - val_accuracy: 0.7446 - val_loss: 0.8222 - learning_rate: 0.0010
Epoch 2/20
533/533 ━━━━━━━━━━━━━━━━━━━━ 12s 23ms/step - accuracy: 0.8470 - loss: 0.3348 - val_accuracy: 0.7898 - val_loss: 0.6706 - learning_rate: 0.0010
Epoch 3/20
533/533 ━━━━━━━━━━━━━━━━━━━━ 12s 23ms/step - accuracy: 0.8662 - loss: 0.2957 - val_accuracy: 0.7706 - val_loss: 0.9371 - learning_rate: 0.0010
Epoch 4/20
533/533 ━━━━━━━━━━━━━━━━━━━━ 13s 25ms/step - accuracy: 0.8787 - loss: 0.2682 - val_accuracy: 0.7720 - val_loss: 0.9775 - learning_rate: 0.0010
Epoch 5/20
533/533 ━━━━━━━━━━━━━━━━━━━━ 13s 24ms/step - accuracy: 0.8929 - loss: 0.2381 - val_accuracy: 0.7683 - val_loss: 0.9902 - learning_rate: 5.0000e-04
Epoch 6/20
533/533 ━━━━━━━━━━━━━━━━━━━━ 14s 26ms/step - accuracy: 0.8986 - loss: 0.2254 - val_accuracy: 0.7791 - val_loss: 0.7436 - learning_rate: 5.0000e-04
Epoch 7/20
533/533 ━━━━━━━━━━━━━━━━━━━━ 11s 21ms/step - accuracy: 0.

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

cnn_pred = (
    cnn_model.predict(X_test_dl)
    > 0.5
).astype(int).flatten()

cnn_results = {
    "Model": "CNN",
    "Accuracy": accuracy_score(y_test_dl, cnn_pred),
    "Precision": precision_score(y_test_dl, cnn_pred),
    "Recall": recall_score(y_test_dl, cnn_pred),
    "F1 Score": f1_score(y_test_dl, cnn_pred)
}

print(cnn_results)

536/536 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step
{'Model': 'CNN', 'Accuracy': 0.7898330804248862, 'Precision': 0.8323965898326492, 'Recall': 0.9329699886749717, 'F1 Score': 0.8798184427460535}


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    LSTM,
    Dense,
    Dropout
)

lstm_model = Sequential([

    LSTM(
        64,
        return_sequences=True,
        input_shape=(128,17)
    ),

    Dropout(0.3),

    LSTM(32),

    Dropout(0.3),

    Dense(16, activation='relu'),

    Dense(1, activation='sigmoid')
])

lstm_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

lstm_model.summary()

d:\VIT_Projects\amigos\amvenv\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 128, 64)        │        20,992 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 33,953 (132.63 KB)

 Trainable params: 33,953 (132.63 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
lstm_callbacks = [

    EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    ),

    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2
    ),

    ModelCheckpoint(
        "Models/lstm_best.keras",
        save_best_only=True,
        monitor="val_accuracy"
    )
]

In [ ]:
lstm_history = lstm_model.fit(
    X_train_dl,
    y_train_dl,
    validation_data=(X_test_dl, y_test_dl),
    epochs=20,
    batch_size=256,
    callbacks=lstm_callbacks
)

Epoch 1/20
533/533 ━━━━━━━━━━━━━━━━━━━━ 63s 114ms/step - accuracy: 0.7903 - loss: 0.4814 - val_accuracy: 0.7869 - val_loss: 0.6130 - learning_rate: 0.0010
Epoch 2/20
533/533 ━━━━━━━━━━━━━━━━━━━━ 58s 108ms/step - accuracy: 0.8138 - loss: 0.4265 - val_accuracy: 0.7616 - val_loss: 0.7091 - learning_rate: 0.0010
Epoch 3/20
533/533 ━━━━━━━━━━━━━━━━━━━━ 63s 118ms/step - accuracy: 0.8250 - loss: 0.3797 - val_accuracy: 0.7326 - val_loss: 0.8070 - learning_rate: 0.0010
Epoch 4/20
533/533 ━━━━━━━━━━━━━━━━━━━━ 60s 113ms/step - accuracy: 0.8408 - loss: 0.3406 - val_accuracy: 0.7385 - val_loss: 0.8164 - learning_rate: 5.0000e-04
Epoch 5/20
533/533 ━━━━━━━━━━━━━━━━━━━━ 56s 106ms/step - accuracy: 0.8487 - loss: 0.3286 - val_accuracy: 0.7536 - val_loss: 0.7692 - learning_rate: 5.0000e-04
Epoch 6/20
533/533 ━━━━━━━━━━━━━━━━━━━━ 58s 108ms/step - accuracy: 0.8563 - loss: 0.3121 - val_accuracy: 0.7473 - val_loss: 0.8296 - learning_rate: 2.5000e-04


In [ ]:
lstm_pred = (
    lstm_model.predict(X_test_dl)
    > 0.5
).astype(int).flatten()

lstm_results = {
    "Model": "LSTM",
    "Accuracy": accuracy_score(y_test_dl, lstm_pred),
    "Precision": precision_score(y_test_dl, lstm_pred),
    "Recall": recall_score(y_test_dl, lstm_pred),
    "F1 Score": f1_score(y_test_dl, lstm_pred)
}

print(lstm_results)

536/536 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step
{'Model': 'LSTM', 'Accuracy': 0.7869149060347846, 'Precision': 0.8264473110238674, 'Recall': 0.9387032842582106, 'F1 Score': 0.8790057995028997}


In [ ]:
dl_df = pd.DataFrame([
    cnn_results,
    lstm_results
])

final_comparison = pd.concat(
    [comparison_df, dl_df],
    ignore_index=True
)

final_comparison

,Model,Accuracy,Precision,Recall,F1 Score
0,Random Forest,0.800981,0.702545,0.800981,0.740388
1,Linear SVM,0.783880,0.724143,0.783880,0.747062
2,CNN,0.789833,0.832397,0.932970,0.879818
3,LSTM,0.786915,0.826447,0.938703,0.879006
